In [43]:
import os, json, random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import evaluate

In [44]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(DEVICE)
TRAIN_CSV = "data/train_sent_emo.csv"
DEV_CSV = "data/dev_sent_emo.csv"
TEST_CSV = "data/test_sent_emo.csv"
TEXT_COL = "Utterance"
LABEL_COL = "Emotion"
MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = "out/deberta_v3_base_emotion"
EPOCHS = 5
LR = 2e-5
BATCH_SIZE = 16
MAX_LENGTH = 64
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
os.makedirs(OUTPUT_DIR, exist_ok=True)


mps


In [45]:
data_files = {"train": TRAIN_CSV, "validation": DEV_CSV, "test": TEST_CSV}
ds = load_dataset("csv", data_files=data_files)
for split in ["train","validation","test"]:
    cols = set(ds[split].column_names)
    if TEXT_COL not in cols or LABEL_COL not in cols:
        raise ValueError(f"Missing column in {split}: expected {TEXT_COL} and {LABEL_COL}, got {cols}")
    ds[split] = ds[split].filter(lambda x: x[TEXT_COL] is not None and x[LABEL_COL] is not None and str(x[TEXT_COL]).strip()!="")
for split in ["train","validation","test"]:
    print(split, len(ds[split]))


train 9989
validation 1109
test 2610


In [46]:
unique_labels = sorted(list(set(ds["train"][LABEL_COL])))
label2id = {l:i for i,l in enumerate(unique_labels)}
id2label = {i:l for l,i in label2id.items()}
print(label2id)


{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


In [47]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess(examples):
    enc = tokenizer(examples[TEXT_COL], truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = [label2id[x] for x in examples[LABEL_COL]]
    return enc
encoded = DatasetDict({
    "train": ds["train"].map(preprocess, batched=True, remove_columns=ds["train"].column_names),
    "validation": ds["validation"].map(preprocess, batched=True, remove_columns=ds["validation"].column_names),
    "test": ds["test"].map(preprocess, batched=True, remove_columns=ds["test"].column_names),
})
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [48]:
y_train = [label2id[y] for y in ds["train"][LABEL_COL]]
classes = np.arange(len(unique_labels))
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print(class_weights.tolist())


[1.2867448329925537, 5.265682697296143, 5.324626922607422, 0.8187034130096436, 0.3029724061489105, 2.0893118381500244, 1.1842323541641235]


In [49]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
)
# Let Trainer manage device placement (MPS/CPU)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total_params)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


184427527


In [50]:
metric_f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": macro_f1}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        cw = self.class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [51]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",  # Correct parameter name for this Transformers version
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,  # explicit for clarity
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=False,
    use_mps_device=True,  # ensure Trainer uses Apple Silicon MPS
)
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights
)
train_output = trainer.train()
print(train_output)
eval_metrics = trainer.evaluate()
print(eval_metrics)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/training_args.py:2278: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
/var/folders/r6/rxbbw7bx48x8xjgx9cmwq_180000gn/T/ipykernel_11630/756251230.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: Us

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.523900,1.520777,0.596032,0.416429
2,1.438200,1.382769,0.548242,0.446744
3,1.065500,1.440720,0.578900,0.462881
4,0.815600,1.492685,0.605951,0.493702
5,0.684600,1.586502,0.595131,0.487492


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pin

TrainOutput(global_step=3125, training_loss=1.1644505236816407, metrics={'train_runtime': 793.6177, 'train_samples_per_second': 62.933, 'train_steps_per_second': 3.938, 'total_flos': 811055322650286.0, 'train_loss': 1.1644505236816407, 'epoch': 5.0})


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.4926851987838745, 'eval_accuracy': 0.60595130748422, 'eval_f1': 0.4937020980645027, 'eval_runtime': 3.6472, 'eval_samples_per_second': 304.073, 'eval_steps_per_second': 19.193, 'epoch': 5.0}


In [52]:
pred = trainer.predict(encoded["test"])
preds = np.argmax(pred.predictions, axis=-1)
report = classification_report(pred.label_ids, preds, target_names=[id2label[i] for i in range(len(unique_labels))])
print(report)
cm = confusion_matrix(pred.label_ids, preds, labels=list(range(len(unique_labels))))
print(cm)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

       anger       0.46      0.46      0.46       345
     disgust       0.40      0.35      0.38        68
        fear       0.15      0.36      0.21        50
         joy       0.60      0.61      0.60       402
     neutral       0.80      0.71      0.75      1256
     sadness       0.38      0.36      0.37       208
    surprise       0.48      0.64      0.55       281

    accuracy                           0.61      2610
   macro avg       0.47      0.50      0.47      2610
weighted avg       0.64      0.61      0.62      2610

[[159  16  16  34  47  21  52]
 [ 13  24   3   1  16   4   7]
 [  7   1  18   4   9   7   4]
 [ 34   6   9 244  65   6  38]
 [ 66  10  53  85 886  76  80]
 [ 34   2  16  11  54  75  16]
 [ 35   1   7  27  24   6 181]]


In [53]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "labels.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)


In [55]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
pipe_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
pipe_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
if DEVICE == "mps":
    pipe_model.to("mps")
texts = [
    "I'm excited about the math quiz!",
    "I feel really down today.",
    "This is making me angry.",
    "I'm worried about tomorrow's exam.",
    "That was yuck.",
    "Wow, I didn't expect that!",
    "It's okay, I guess."
]
with torch.no_grad():
    for t in texts:
        inputs = pipe_tok(t, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
        if DEVICE == "mps":
            inputs = {k:v.to("mps") for k,v in inputs.items()}
        logits = pipe_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        pred_id = int(probs.argmax())
        print(t, id2label[pred_id], float(probs[pred_id]))


I'm excited about the math quiz! joy 0.9449805021286011
I feel really down today. sadness 0.9735084176063538
This is making me angry. anger 0.6968483924865723
I'm worried about tomorrow's exam. fear 0.9503266215324402
That was yuck. disgust 0.9032173752784729
Wow, I didn't expect that! surprise 0.9664970636367798
It's okay, I guess. neutral 0.5728791952133179
